Computer Vision Based Disease Classfication

In [ ]:
# Install required libraries (run once)

!pip install -q kagglehub
!pip install -q torch torchvision
!pip install -q scikit-learn
!pip install -q matplotlib
!pip install -q seaborn

# Core Deep Learning
import torch
import torch.nn as nn
import torch.optim as optim

# Image Processing
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# Numerical Computation
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Evaluation Metrics
from sklearn.metrics import confusion_matrix, classification_report
from collections import Counter




In [32]:
import kagglehub
import os

# Download dataset from Kaggle
dataset_path = kagglehub.dataset_download(
    "tawsifurrahman/covid19-radiography-database"
)

print("Raw Dataset Path:", dataset_path)

# Check folder structure
print("Initial Contents:", os.listdir(dataset_path))

# Fix folder level if nested
if "COVID-19_Radiography_Dataset" in os.listdir(dataset_path):
    dataset_path = os.path.join(dataset_path, "COVID-19_Radiography_Dataset")

print("Final Dataset Path:", dataset_path)
print("Final Contents:", os.listdir(dataset_path))


Using Colab cache for faster access to the 'covid19-radiography-database' dataset.
Raw Dataset Path: /kaggle/input/covid19-radiography-database
Initial Contents: ['COVID-19_Radiography_Dataset']
Final Dataset Path: /kaggle/input/covid19-radiography-database/COVID-19_Radiography_Dataset
Final Contents: ['Lung_Opacity.metadata.xlsx', 'Normal.metadata.xlsx', 'README.md.txt', 'COVID.metadata.xlsx', 'Normal', 'Lung_Opacity', 'Viral Pneumonia.metadata.xlsx', 'Viral Pneumonia', 'COVID']


Dataset Path Verification Block

In [ ]:


print("Verifying dataset path...\n")
print("Dataset Path:", dataset_path)

# List all items
items = os.listdir(dataset_path)
print("\nAll items inside dataset folder:")
for item in items:
    print(" -", item)

# Detect only directories (class folders)
class_folders = [
    folder for folder in items
    if os.path.isdir(os.path.join(dataset_path, folder))
]

print("\nDetected class folders:")
for folder in class_folders:
    print(" -", folder)

print("\nNumber of class folders detected:", len(class_folders))


Verifying dataset path...

Dataset Path: /kaggle/input/covid19-radiography-database/COVID-19_Radiography_Dataset

All items inside dataset folder:
 - Lung_Opacity.metadata.xlsx
 - Normal.metadata.xlsx
 - README.md.txt
 - COVID.metadata.xlsx
 - Normal
 - Lung_Opacity
 - Viral Pneumonia.metadata.xlsx
 - Viral Pneumonia
 - COVID

Detected class folders:
 - Normal
 - Lung_Opacity
 - Viral Pneumonia
 - COVID

Number of class folders detected: 4


In [35]:
# Detect available device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Selected Device:", device)

# If GPU is available, print GPU details
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("CUDA Version:", torch.version.cuda)
else:
    print("Running on CPU")


Selected Device: cuda
GPU Name: Tesla T4
CUDA Version: 12.8


Image Transform / Preprocessing Block

In [36]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),              # Standardize image size
    transforms.Grayscale(num_output_channels=1),# Convert to single-channel (X-ray is grayscale)
    transforms.ToTensor(),                      # Convert image to tensor [0,1]
    transforms.Normalize((0.5,), (0.5,))        # Normalize for stable training
])


Dataset Loading Block (ImageFolder)

In [37]:
dataset = datasets.ImageFolder(
    root=dataset_path,
    transform=transform
)

print("Detected Classes:", dataset.classes)
print("Number of Classes:", len(dataset.classes))
print("Total Images:", len(dataset))


Detected Classes: ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
Number of Classes: 4
Total Images: 42330
